# Reset for another run of 08-05c-02 and 08-05c-03

[08-05c-02](08-05c-02-wrapper-container-app.ipynb) ends by revoking user B's `News.Read.Science`, and [08-05c-03](08-05c-03-apim-passthrough.ipynb) tests what that revocation does. To run both again without the teardown and a fresh build, run this notebook after 08-05c-03. It puts the app roles back as [08-05c-01](08-05c-01-identity-and-agent.ipynb) Step 9 left them.

It changes nothing else. 08-05c-02 Step 4 and 08-05c-03 Step 3 each create a new `news-desk-agent` version, and the agent endpoint runs the latest one, so each notebook run from the top tests its own track again. This notebook creates nothing, so it has no cleanup; the teardown in 08-05c-01 still removes the whole lab.

**What this notebook does:**
1. Puts back the app role assignments of 08-05c-01 Step 9 and removes `NEWS_SCIENCE_REVOKED_AT` from `.env`
2. Optionally issues user B a new Temporary Access Pass
3. Lists `news-desk-agent`'s versions and the tool each one uses

## Prerequisites

1. **08-05c-01, 08-05c-02 and 08-05c-03** have run. Run before 08-05c-03, this notebook gives the role back before 08-05c-03 can test its removal.
2. **Azure CLI**: `az login` as the Global Administrator who ran 08-05c-01. Steps 1 and 3 use the CLI token.
3. **Step 2 only**: a device code sign-in as user A, your admin account.

## Step 0: Configuration

In [ ]:
import hashlib, json, os, subprocess
from pathlib import Path

import requests
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

import news_mcp_checks as checks
import news_mcp_identity as ident
from news_mcp_identity import authz

repo_root = Path(subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip())
ENV_FILE = repo_root / '.env'
load_dotenv(ENV_FILE, override=True)

REQUIRED = ['NEWS_MCP_RESOURCE_SP_ID', 'NEWS_DESK_CLIENT_APP_ID', 'NEWS_TEST_USER_UPN', 'NEWS_TEST_USER_ID']
missing = [key for key in REQUIRED if not os.environ.get(key)]
if missing:
    raise SystemExit(f"Missing in .env: {', '.join(missing)}. Run 08-05c-01 first.")

account = json.loads(subprocess.run('az account show -o json', shell=True, capture_output=True, text=True).stdout)
SUBSCRIPTION_ID, TENANT_ID = account['id'], account['tenantId']
SUFFIX = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

RESOURCE_SP_ID = os.environ['NEWS_MCP_RESOURCE_SP_ID']
CLIENT_APP_ID = os.environ['NEWS_DESK_CLIENT_APP_ID']
TEST_USER_ID = os.environ['NEWS_TEST_USER_ID']

cli = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=cli)


def graph(method, path, body=None, token=None):
    """Microsoft Graph v1.0, as the Azure CLI user unless another token is given."""
    token = token or cli.get_token('https://graph.microsoft.com/.default').token
    r = requests.request(method, f'https://graph.microsoft.com/v1.0{path}', json=body,
                         headers={'Authorization': f'Bearer {token}'}, timeout=60)
    if not r.ok:
        raise RuntimeError(f'{method} {path}: HTTP {r.status_code} {r.text[:600]}')
    return r.json() if r.content else None


def env_set(**values):
    """Write keys to the repo .env: replace the key's line if present, append it otherwise."""
    lines = ENV_FILE.read_text().splitlines()
    for key, value in values.items():
        os.environ[key] = value
        at = [i for i, line in enumerate(lines) if line.split('=', 1)[0].strip() == key]
        if at:
            lines[at[0]] = f'{key}={value}'
        else:
            lines.append(f'{key}={value}')
    ENV_FILE.write_text('\n'.join(lines) + '\n')


ADMIN = graph('GET', '/me?$select=id,userPrincipalName')
print(f'Suffix : {SUFFIX}')
print(f"User A : {ADMIN['userPrincipalName']}")
print(f"User B : {os.environ['NEWS_TEST_USER_UPN']}")

## Step 1: Put the app roles back

08-05c-01 Step 9 gave user A `News.Read.All`, and user B `News.Read.Sports`, `News.Read.Science` and `News.Read.Health`. The cell adds whichever of those assignments is missing (after 08-05c-02 Step 9, user B's `News.Read.Science`) and leaves the rest as they are. It then removes `NEWS_SCIENCE_REVOKED_AT` from `.env`, so 08-05c-03 stops at its configuration until 08-05c-02 Step 9 revokes the role again.

Entra puts the role in user B's next token for `news-mcp-{suffix}`; a token issued earlier keeps the roles it had until it expires. If 08-05c-02's science case comes back `hidden` for user B, Foundry is still presenting such a token: sign user B in again and re-run the matrix.

In [ ]:
WANTED = {ADMIN['id']: ['News.Read.All'], TEST_USER_ID: ['News.Read.Sports', 'News.Read.Science', 'News.Read.Health']}
LABELS = {ADMIN['id']: 'user A (you)', TEST_USER_ID: 'user B (news-sports-desk)'}
ROLE_BY_ID = {authz.permission_id(SUFFIX, role): role for role in authz.ROLES}

assigned_to = f'/servicePrincipals/{RESOURCE_SP_ID}/appRoleAssignedTo'
have = {(a['principalId'], a['appRoleId']) for a in graph('GET', assigned_to)['value']}
for principal, roles in WANTED.items():
    for role in roles:
        body = ident.role_assignment_body(SUFFIX, principal, RESOURCE_SP_ID, role)
        if (principal, body['appRoleId']) not in have:
            graph('POST', assigned_to, body)
            print(f'Added {role} for {LABELS[principal]}')

lines = ENV_FILE.read_text().splitlines()
ENV_FILE.write_text('\n'.join(l for l in lines if l.split('=', 1)[0].strip() != 'NEWS_SCIENCE_REVOKED_AT') + '\n')
os.environ.pop('NEWS_SCIENCE_REVOKED_AT', None)
print('Removed NEWS_SCIENCE_REVOKED_AT from .env')

print(f'\nApp role assignments on news-mcp-{SUFFIX}:')
for a in graph('GET', assigned_to)['value']:
    print(f"  {LABELS.get(a['principalId'], a['principalDisplayName']):<28} {ROLE_BY_ID[a['appRoleId']]}")

## Step 2 (optional): A new pass for user B

User B's pass lasts 8 hours from when it was issued: 08-05c-01 Step 5, or the last run of this step. Tokens from a pass sign-in end when the pass does. If the pass has expired, or will before you finish 08-05c-03, set `NEW_PASS = True` and run the cell.

Issuing a pass needs the Graph permission `UserAuthenticationMethod.ReadWrite.All`, which the Azure CLI token does not carry. The cell signs you in through `news-desk-client`, as 08-05c-01 Step 3 does: open the address it prints, enter its code and sign in as **user A**, your admin account (the one `az login` uses), with your usual password and multifactor method. The cell stops if the token is for another account. It then deletes user B's current pass, issues a new one and writes it to `.env` as `NEWS_TEST_USER_TAP`. The pass is never printed. 08-05c-02 and 08-05c-03 sign user B in with it.

In [ ]:
NEW_PASS = False  # True: sign in as user A and replace user B's pass

if NEW_PASS:
    print(f"User A, your admin account {ADMIN['userPrincipalName']}, with your usual password and multifactor method:")
    admin_cred = checks.user_credential(TENANT_ID, CLIENT_APP_ID, device_code=True)
    admin_token = admin_cred.get_token('https://graph.microsoft.com/UserAuthenticationMethod.ReadWrite.All').token
    if checks.jwt_claims(admin_token)['oid'] != ADMIN['id']:
        raise RuntimeError('That sign-in was not the account the Azure CLI uses. Sign in as user A, your admin account, and re-run this cell.')

    # A user holds at most one pass: replace the current one.
    methods = f'/users/{TEST_USER_ID}/authentication/temporaryAccessPassMethods'
    for old in graph('GET', methods, token=admin_token)['value']:
        graph('DELETE', f"{methods}/{old['id']}", token=admin_token)
    tap = graph('POST', methods, ident.tap_method_body(), token=admin_token)
    env_set(NEWS_TEST_USER_TAP=tap['temporaryAccessPass'])
    print(f"Pass written to .env as NEWS_TEST_USER_TAP: multi-use={not tap['isUsableOnce']}, "
          f"{tap['lifetimeInMinutes']} minutes from {tap['startDateTime']}")
else:
    print('User B keeps the current pass. Set NEW_PASS = True for a new one.')

## Step 3: The agent's versions

The agent endpoint runs the latest version of `news-desk-agent`. After 08-05c-03 that is the Track B version, with the `news_mcp_apim` tool. 08-05c-02 Step 4 adds a Track A version, with `news_mcp_wrapper`, and makes it the latest; 08-05c-03 Step 3 then adds a Track B version again.

Next: run 08-05c-02 from the top, then 08-05c-03 within the hour after 08-05c-02 Step 9.

In [ ]:
versions = sorted((v.as_dict() for v in project.agents.list_versions(agent_name=ident.AGENT_NAME)),
                  key=lambda v: int(v['version']))
for v in versions:
    tools = [t['server_label'] for t in v['definition'].get('tools', [])]
    print(f"  v{v['version']:<3} {', '.join(tools) or 'no tools'}")
print(f"\nThe agent endpoint runs v{versions[-1]['version']}.")